In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # 01 · Dataset Setup
# MAGIC Run this notebook first (the other two notebooks call it with `%run`).
# MAGIC Everything is created **inline** – no file uploads needed.
# MAGIC
# MAGIC | DataFrame / temp view | What it's for |
# MAGIC |---|---|
# MAGIC | `employees` | joins, window functions, self-join, aggregations (has ties, NULL dept, manager hierarchy) |
# MAGIC | `departments` | joins (one dept has no employees) |
# MAGIC | `customers`, `orders`, `order_items`, `products` | e-commerce: multi-table joins, revenue, MoM growth, running totals |
# MAGIC | `login_activity` | consecutive-days streak (gaps & islands) |
# MAGIC | `web_events` | sessionization |
# MAGIC | `stock_prices` | lag/lead, running max, consecutive increases |
# MAGIC | `dirty_customers` | null handling, cleaning, dedupe, date parsing |
# MAGIC | `json_events` | JSON parsing, explode, arrays, structs |
# MAGIC | `text_lines` | word count |
# MAGIC | `big_events`, `dim_small` | skew / salting / broadcast / partition practice |
# MAGIC
# MAGIC Every DataFrame is also registered as a **temp view** with the same name, so you can use `spark.sql(...)` too.

In [0]:

from pyspark.sql import functions as F, Window
from pyspark.sql.types import *

# Writable location for the file/Delta exercises.
# If your workspace blocks DBFS root, switch to a Unity Catalog Volume, e.g. "/Volumes/<catalog>/<schema>/<volume>/pyspark_practice"
BASE_PATH = "/tmp/pyspark_practice"

def register(name, df):
    df.createOrReplaceTempView(name)
    return df

# COMMAND ----------

# MAGIC %md
# MAGIC ## HR data

# COMMAND ----------

emp_schema = "emp_id INT, emp_name STRING, dept_id INT, salary INT, manager_id INT, hire_date STRING, city STRING, age INT"

employees = register("employees", spark.createDataFrame([
    (1,  "Arjun",  10, 120000, None, "2016-01-10", "Bengaluru", 41),
    (2,  "Bhavna", 10,  95000, 1,    "2017-05-21", "Bengaluru", 36),
    (3,  "Chirag", 20,  95000, 1,    "2018-07-15", "Mumbai",    33),
    (4,  "Divya",  20,  70000, 3,    "2019-02-01", "Mumbai",    29),
    (5,  "Esha",   30,  65000, 3,    "2020-11-30", "Delhi",     27),
    (6,  "Farhan", 10,  88000, 2,    "2019-08-19", "Bengaluru", 31),
    (7,  "Gita",   30,  72000, 5,    "2021-03-03", "Delhi",     30),
    (8,  "Harsh",  20,  70000, 3,    "2021-09-09", "Pune",      28),
    (9,  "Isha",   10,  60000, 2,    "2022-01-17", "Bengaluru", 26),
    (10, "Jatin",  40,  55000, 1,    "2022-06-25", "Hyderabad", 25),
    (11, "Kavya",  30,  72000, 5,    "2023-04-10", "Delhi",     28),
    (12, "Lokesh", 10,  88000, 2,    "2020-10-12", "Chennai",   35),
    (13, "Meera",  None, 50000, 1,   "2023-08-01", "Pune",      24),
    (14, "Nikhil", 20,  45000, 4,    "2024-01-15", "Mumbai",    23),
], emp_schema).withColumn("hire_date", F.to_date("hire_date")))

departments = register("departments", spark.createDataFrame([
    (10, "Engineering", "Bengaluru"),
    (20, "Sales",       "Mumbai"),
    (30, "Marketing",   "Delhi"),
    (40, "HR",          "Hyderabad"),
    (50, "Finance",     "Mumbai"),     # no employees
], "dept_id INT, dept_name STRING, location STRING"))

# COMMAND ----------

# MAGIC %md
# MAGIC ## E-commerce data (order_items are independent of orders.amount – don't try to reconcile them)

# COMMAND ----------

customers = register("customers", spark.createDataFrame([
    (1, "Rohan",  "rohan@gmail.com",   "Bengaluru", "2023-01-05"),
    (2, "Sneha",  "sneha@yahoo.com",   "Mumbai",    "2023-02-14"),
    (3, "Vikram", "vikram@gmail.com",  "Delhi",     "2023-03-20"),
    (4, "Anita",  "anita@outlook.com", "Pune",      "2023-05-11"),
    (5, "Rahul",  "rahul@gmail.com",   "Chennai",   "2023-07-01"),
    (6, "Pooja",  "pooja@yahoo.com",   "Bengaluru", "2023-09-09"),
    (7, "Sanjay", "sanjay@gmail.com",  "Hyderabad", "2024-01-02"),   # never ordered
    (8, "Tara",   "tara@outlook.com",  "Mumbai",    "2024-02-18"),   # never ordered
], "customer_id INT, customer_name STRING, email STRING, city STRING, signup_date STRING")
 .withColumn("signup_date", F.to_date("signup_date")))

orders = register("orders", spark.createDataFrame([
    (101, 1, "2024-01-05", 2500.0, "DELIVERED"),
    (102, 2, "2024-01-12", 1200.0, "DELIVERED"),
    (103, 1, "2024-01-28",  800.0, "CANCELLED"),
    (104, 3, "2024-02-03", 4500.0, "DELIVERED"),
    (105, 4, "2024-02-14",  300.0, "RETURNED"),
    (106, 2, "2024-02-20", 2200.0, "DELIVERED"),
    (107, 1, "2024-03-02", 1500.0, "DELIVERED"),
    (108, 5, "2024-03-15", 5200.0, "DELIVERED"),
    (109, 3, "2024-03-18",  750.0, "DELIVERED"),
    (110, 6, "2024-03-29", 1800.0, "SHIPPED"),
    (111, 2, "2024-04-04",  950.0, "DELIVERED"),
    (112, 1, "2024-04-19", 3100.0, "DELIVERED"),
    (113, 5, "2024-05-07", 2700.0, "DELIVERED"),
    (114, 3, "2024-05-21", 1300.0, "CANCELLED"),
    (115, 4, "2024-06-10", 2000.0, "DELIVERED"),
], "order_id INT, customer_id INT, order_date STRING, amount DOUBLE, status STRING")
 .withColumn("order_date", F.to_date("order_date")))

products = register("products", spark.createDataFrame([
    (1, "Mouse",      "Accessories",  500.0),
    (2, "Keyboard",   "Accessories", 1200.0),
    (3, "Headphones", "Electronics", 2500.0),
    (4, "Monitor",    "Electronics", 9000.0),
    (5, "Notebook",   "Stationery",   150.0),
    (6, "Pen Pack",   "Stationery",    80.0),
], "product_id INT, product_name STRING, category STRING, price DOUBLE"))

order_items = register("order_items", spark.createDataFrame([
    (101, 3, 1), (101, 1, 2), (102, 2, 1), (103, 5, 5), (104, 4, 2), (105, 6, 3),
    (106, 3, 1), (106, 2, 1), (107, 1, 3), (108, 4, 1), (109, 5, 4), (110, 3, 2),
    (111, 6, 6), (112, 4, 1), (113, 2, 2), (114, 1, 2), (115, 3, 1), (115, 5, 2),
], "order_id INT, product_id INT, quantity INT"))

# COMMAND ----------

# MAGIC %md
# MAGIC ## Event / time-series data

# COMMAND ----------

login_activity = register("login_activity", spark.createDataFrame([
    (1, "2024-01-01"), (1, "2024-01-02"), (1, "2024-01-02"), (1, "2024-01-03"),   # duplicate on 01-02
    (1, "2024-01-05"), (1, "2024-01-06"), (1, "2024-01-09"),
    (2, "2024-01-01"), (2, "2024-01-02"), (2, "2024-01-03"),
    (2, "2024-01-04"), (2, "2024-01-05"), (2, "2024-01-06"),
    (3, "2024-01-02"), (3, "2024-01-04"), (3, "2024-01-06"),
], "user_id INT, login_date STRING").withColumn("login_date", F.to_date("login_date")))

web_events = register("web_events", spark.createDataFrame([
    (1, "2024-03-01 10:00:00", "page_view",   "home"),
    (1, "2024-03-01 10:05:00", "click",       "product_12"),
    (1, "2024-03-01 10:20:00", "add_to_cart", "product_12"),
    (1, "2024-03-01 11:30:00", "page_view",   "home"),
    (1, "2024-03-01 11:35:00", "purchase",    "checkout"),
    (2, "2024-03-01 09:00:00", "page_view",   "home"),
    (2, "2024-03-01 09:50:00", "page_view",   "search"),
    (2, "2024-03-01 09:55:00", "click",       "product_7"),
    (2, "2024-03-01 12:00:00", "page_view",   "home"),
    (3, "2024-03-01 15:00:00", "page_view",   "home"),
], "user_id INT, event_time STRING, event_type STRING, page STRING")
 .withColumn("event_time", F.to_timestamp("event_time")))

stock_prices = register("stock_prices", spark.createDataFrame([
    ("AAPL", "2024-03-01", 180.0), ("AAPL", "2024-03-04", 182.5), ("AAPL", "2024-03-05", 181.0),
    ("AAPL", "2024-03-06", 183.2), ("AAPL", "2024-03-07", 185.0), ("AAPL", "2024-03-08", 187.4),
    ("GOOG", "2024-03-01", 140.0), ("GOOG", "2024-03-04", 139.0), ("GOOG", "2024-03-05", 141.5),
    ("GOOG", "2024-03-06", 143.0), ("GOOG", "2024-03-07", 142.0), ("GOOG", "2024-03-08", 144.5),
], "symbol STRING, trade_date STRING, close DOUBLE").withColumn("trade_date", F.to_date("trade_date")))

# COMMAND ----------

# MAGIC %md
# MAGIC ## Messy / semi-structured data

# COMMAND ----------

dirty_customers = register("dirty_customers", spark.createDataFrame([
    (1, " alice SMITH ", "ALICE@Gmail.com",       "98765-43210",     "12/03/1995",  "2024-01-01"),
    (1, " alice SMITH ", "alice.smith@gmail.com", "+91 98765 43210", "12/03/1995",  "2024-03-01"),
    (2, "bob  marley",   "bob@gmail",             "9876543211",      "1990-07-21",  "2024-02-10"),
    (3, None,            "carol@yahoo.com",       None,              "21-Jul-1988", "2024-02-11"),
    (4, "DAVE",          "dave@outlook.com",      "(987) 654-3212",  "1992/11/05",  "2024-01-15"),
    (5, "eve adams",     "",                      "9876543213",      None,          "2024-01-20"),
    (5, "eve adams",     "",                      "9876543213",      None,          "2024-01-20"),  # exact duplicate
    (6, "Frank",         "frank@@gmail.com",      "123",             "1985-12-31",  "2024-03-05"),
], "id INT, name STRING, email STRING, phone STRING, dob STRING, updated_at STRING"))

json_events = register("json_events", spark.createDataFrame([
    (1, '{"order_id": 1, "customer": {"id": 10, "name": "Asha"}, "items": [{"sku": "A1", "qty": 2}, {"sku": "B2", "qty": 1}], "tags": ["new", "vip"]}'),
    (2, '{"order_id": 2, "customer": {"id": 11, "name": "Bala"}, "items": [{"sku": "C3", "qty": 5}], "tags": []}'),
], "id INT, payload STRING"))

text_lines = register("text_lines", spark.createDataFrame([
    ("spark is fast",),
    ("spark is simple and spark is popular",),
    ("pyspark makes spark easy",),
], "line STRING"))

# COMMAND ----------

# MAGIC %md
# MAGIC ## Larger / skewed data (for performance exercises)
# MAGIC `key = 1` holds ~80% of the rows → classic data-skew scenario.

# COMMAND ----------

BIG_N = 5_000_000

big_events = register("big_events", (
    spark.range(BIG_N)
    .withColumn("key", F.when(F.col("id") % 10 < 8, F.lit(1)).otherwise(F.col("id") % 1000))
    .withColumn("amount", (F.rand(seed=42) * 100).cast("double"))
))

dim_small = register("dim_small", (
    spark.range(1000).withColumnRenamed("id", "key")
    .withColumn("label", F.concat(F.lit("key_"), F.col("key").cast("string")))
))

# COMMAND ----------

for name in ["employees", "departments", "customers", "orders", "order_items", "products",
             "login_activity", "web_events", "stock_prices", "dirty_customers", "json_events", "text_lines"]:
    print(f"{name:16s} rows = {spark.table(name).count()}")
print("big_events / dim_small are lazy (not counted).")

# 03 · Solutions
Try the questions in `02_practice_questions` first – peek only when stuck.

Run the dataset setup cell above first so all DataFrames and temp views exist.

In [0]:
from pyspark.sql import functions as F, Window

## 1 · Basics & column transformations

**Q1.** ⭐  Select `emp_name` and `salary`. Rename `salary` to `annual_ctc` and add `monthly_salary` (= annual / 12, rounded to 2 decimals).

In [0]:
employees.select(
    "emp_name",
    F.col("salary").alias("annual_ctc"),
    F.round(F.col("salary") / 12, 2).alias("monthly_salary"),
).show()

**Q2.** ⭐  Filter employees who work in **Bengaluru or Mumbai**, earn **more than 70,000** and are **younger than 40**.

In [0]:
employees.filter(
    F.col("city").isin("Bengaluru", "Mumbai") & (F.col("salary") > 70000) & (F.col("age") < 40)
).show()

**Q3.** ⭐  Add a `grade` column: salary >= 100000 → 'A', >= 80000 → 'B', >= 60000 → 'C', otherwise 'D'.

In [0]:
employees.withColumn(
    "grade",
    F.when(F.col("salary") >= 100000, "A")
     .when(F.col("salary") >= 80000, "B")
     .when(F.col("salary") >= 60000, "C")
     .otherwise("D"),
).select("emp_name", "salary", "grade").show()

**Q4.** ⭐  Add `hire_year` and `tenure_years` (years between `hire_date` and today, 1 decimal).

In [0]:
employees.withColumn("hire_year", F.year("hire_date")) \
    .withColumn("tenure_years", F.round(F.months_between(F.current_date(), "hire_date") / 12, 1)) \
    .select("emp_name", "hire_date", "hire_year", "tenure_years").show()

**Q5.** ⭐  Rename `emp_name` → `name`, cast `salary` to double, drop `age`. Do the same using `selectExpr`.

In [0]:
df = (employees.withColumnRenamed("emp_name", "name")
      .withColumn("salary", F.col("salary").cast("double"))
      .drop("age"))
df.printSchema()

employees.selectExpr("emp_id", "emp_name AS name", "CAST(salary AS DOUBLE) AS salary").show(3)

**Q6.** ⭐  Sort by `dept_id` ascending (NULLs last) and then `salary` descending.

In [0]:
employees.orderBy(F.col("dept_id").asc_nulls_last(), F.col("salary").desc()).show()

**Q7.** ⭐  Count distinct cities. Then drop duplicate rows considering only (`dept_id`, `salary`).

In [0]:
print(employees.select("city").distinct().count())
employees.select(F.countDistinct("city")).show()
employees.dropDuplicates(["dept_id", "salary"]).show()

## 2 · Aggregations

**Q8.** ⭐  Per department: employee count, average, min and max salary.

In [0]:
employees.groupBy("dept_id").agg(
    F.count("*").alias("emp_count"),
    F.round(F.avg("salary"), 2).alias("avg_salary"),
    F.min("salary").alias("min_salary"),
    F.max("salary").alias("max_salary"),
).orderBy("dept_id").show()

**Q9.** ⭐⭐  Departments with **more than 2 employees** and **average salary >= 70,000** (HAVING equivalent).

In [0]:
(employees.groupBy("dept_id")
    .agg(F.count("*").alias("cnt"), F.avg("salary").alias("avg_sal"))
    .filter((F.col("cnt") > 2) & (F.col("avg_sal") >= 70000))
    .show())

**Q10.** ⭐  Monthly revenue (`yyyy-MM`) for **DELIVERED** orders only, sorted by month.

In [0]:
(orders.filter(F.col("status") == "DELIVERED")
    .groupBy(F.date_format("order_date", "yyyy-MM").alias("month"))
    .agg(F.sum("amount").alias("revenue"))
    .orderBy("month").show())

**Q11.** ⭐⭐  **Conditional aggregation:** one row per customer with the count of DELIVERED, CANCELLED and RETURNED orders as separate columns.

In [0]:
(orders.groupBy("customer_id").agg(
    *[F.sum(F.when(F.col("status") == s, 1).otherwise(0)).alias(s.lower())
      for s in ["DELIVERED", "CANCELLED", "RETURNED"]]
).orderBy("customer_id").show())

**Q12.** ⭐⭐  Each city's salary as a **% of total company salary** (no joins – use a window).

In [0]:
(employees.groupBy("city").agg(F.sum("salary").alias("city_salary"))
    .withColumn("pct_of_total",
                F.round(F.col("city_salary") * 100 / F.sum("city_salary").over(Window.partitionBy()), 2))
    .orderBy(F.desc("pct_of_total")).show())

**Q13.** ⭐⭐  For each department, produce a **sorted list of employee names** (`collect_list`) and a set of distinct cities (`collect_set`).

In [0]:
employees.groupBy("dept_id").agg(
    F.sort_array(F.collect_list("emp_name")).alias("members"),
    F.sort_array(F.collect_set("city")).alias("cities"),
).show(truncate=False)

## 3 · Joins

**Q14.** ⭐  Inner join employees with departments → `emp_name`, `dept_name`.

In [0]:
employees.join(departments, "dept_id").select("emp_name", "dept_name").show()

**Q15.** ⭐  Left join: all employees with their department name; employees without a department should show **'Unassigned'**.

In [0]:
(employees.join(departments, "dept_id", "left")
    .select("emp_name", F.coalesce("dept_name", F.lit("Unassigned")).alias("dept_name")).show())

**Q16.** ⭐  Departments that have **no employees** (anti join).

In [0]:
departments.join(employees, "dept_id", "left_anti").show()

**Q17.** ⭐  Customers who have **never placed an order**.

In [0]:
customers.join(orders, "customer_id", "left_anti").show()

**Q18.** ⭐⭐  **Self join:** each employee with their manager's name (top boss should show NULL manager).

In [0]:
e, m = employees.alias("e"), employees.alias("m")
(e.join(m, F.col("e.manager_id") == F.col("m.emp_id"), "left")
   .select(F.col("e.emp_name").alias("employee"), F.col("m.emp_name").alias("manager")).show())

**Q19.** ⭐⭐  Employees who **earn more than their manager**.

In [0]:
e, m = employees.alias("e"), employees.alias("m")
(e.join(m, F.col("e.manager_id") == F.col("m.emp_id"))
   .filter(F.col("e.salary") > F.col("m.salary"))
   .select(F.col("e.emp_name").alias("employee"), F.col("e.salary"),
           F.col("m.emp_name").alias("manager"), F.col("m.salary").alias("mgr_salary")).show())

**Q20.** ⭐⭐  **Full outer join** employees and departments; show only the rows that are unmatched on either side.

In [0]:
(employees.join(departments, "dept_id", "full")
    .filter(F.col("emp_name").isNull() | F.col("dept_name").isNull())
    .select("dept_id", "emp_name", "dept_name").show())

**Q21.** ⭐⭐⭐  **Multi-table join:** for DELIVERED orders, total spend (quantity × price) per `customer_name` per `category`.

In [0]:
(order_items
    .join(products, "product_id")
    .join(orders.filter(F.col("status") == "DELIVERED"), "order_id")
    .join(customers, "customer_id")
    .withColumn("line_total", F.col("quantity") * F.col("price"))
    .groupBy("customer_name", "category")
    .agg(F.sum("line_total").alias("total_spend"))
    .orderBy("customer_name", "category").show())

**Q22.** ⭐⭐  Customers who placed **at least one DELIVERED order** using a semi join (no duplicates, no extra columns).

In [0]:
customers.join(orders.filter(F.col("status") == "DELIVERED"), "customer_id", "left_semi").show()

**Q23.** ⭐⭐  Join `orders` with `customers` using a **broadcast hint** and inspect the physical plan. What join strategy do you see?

In [0]:
joined = orders.join(F.broadcast(customers), "customer_id")
joined.explain()          # look for BroadcastHashJoin
joined.show(5)

## 4 · Window functions

**Q24.** ⭐⭐  Rank employees by salary **within each department** using `row_number`, `rank` and `dense_rank`. Explain the difference using the ties.

In [0]:
w = Window.partitionBy("dept_id").orderBy(F.col("salary").desc())
(employees.filter(F.col("dept_id").isNotNull())
    .withColumn("row_num", F.row_number().over(w))
    .withColumn("rank", F.rank().over(w))
    .withColumn("dense_rank", F.dense_rank().over(w))
    .select("dept_id", "emp_name", "salary", "row_num", "rank", "dense_rank")
    .orderBy("dept_id", "row_num").show())

**Q25.** ⭐⭐  **Top 2 highest-paid** employees per department (ties for the same salary should all be included).

In [0]:
w = Window.partitionBy("dept_id").orderBy(F.col("salary").desc())
(employees.filter(F.col("dept_id").isNotNull())
    .withColumn("drnk", F.dense_rank().over(w))
    .filter(F.col("drnk") <= 2)
    .orderBy("dept_id", "drnk").show())

**Q26.** ⭐⭐  **Nth highest salary** (company-wide). Find the 2nd highest and make N a variable.

In [0]:
N = 2
w = Window.orderBy(F.col("salary").desc())
(employees.withColumn("drnk", F.dense_rank().over(w))
    .filter(F.col("drnk") == N).select("salary").distinct().show())

# alternative without window
employees.select("salary").distinct().orderBy(F.desc("salary")).limit(N).orderBy("salary").limit(1).show()

**Q27.** ⭐⭐  **Running total** of order amount per customer ordered by date.

In [0]:
w = (Window.partitionBy("customer_id").orderBy("order_date")
     .rowsBetween(Window.unboundedPreceding, Window.currentRow))
(orders.withColumn("running_total", F.sum("amount").over(w))
    .orderBy("customer_id", "order_date").show())

**Q28.** ⭐⭐⭐  **Month-over-month revenue growth %** (DELIVERED orders) using `lag`.

In [0]:
monthly = (orders.filter(F.col("status") == "DELIVERED")
    .groupBy(F.date_format("order_date", "yyyy-MM").alias("month"))
    .agg(F.sum("amount").alias("revenue")))
w = Window.orderBy("month")
(monthly.withColumn("prev_revenue", F.lag("revenue").over(w))
    .withColumn("mom_growth_pct",
                F.round((F.col("revenue") - F.col("prev_revenue")) * 100 / F.col("prev_revenue"), 2))
    .orderBy("month").show())

**Q29.** ⭐⭐  For each employee show **difference from department average** and from the **department maximum** salary.

In [0]:
w = Window.partitionBy("dept_id")
(employees.filter(F.col("dept_id").isNotNull())
    .withColumn("dept_avg", F.round(F.avg("salary").over(w), 2))
    .withColumn("diff_from_avg", F.round(F.col("salary") - F.col("dept_avg"), 2))
    .withColumn("gap_to_max", F.max("salary").over(w) - F.col("salary"))
    .select("dept_id", "emp_name", "salary", "dept_avg", "diff_from_avg", "gap_to_max").show())

**Q30.** ⭐⭐⭐  **3-order moving average** of `amount` per customer (current + 2 previous orders).

In [0]:
w = Window.partitionBy("customer_id").orderBy("order_date").rowsBetween(-2, 0)
(orders.withColumn("moving_avg_3", F.round(F.avg("amount").over(w), 2))
    .orderBy("customer_id", "order_date").show())

**Q31.** ⭐⭐⭐  Show each customer's **first and last order amount** on every row. (Hint: the default window frame will bite you for `last`.)

In [0]:
w = (Window.partitionBy("customer_id").orderBy("order_date")
     .rowsBetween(Window.unboundedPreceding, Window.unboundedFollowing))
(orders.select("customer_id", "order_date", "amount",
               F.first("amount").over(w).alias("first_amount"),
               F.last("amount").over(w).alias("last_amount"))
    .orderBy("customer_id", "order_date").show())

**Q32.** ⭐⭐  **Days since previous order** for each customer (`lag` + `datediff`).

In [0]:
w = Window.partitionBy("customer_id").orderBy("order_date")
(orders.withColumn("prev_order_date", F.lag("order_date").over(w))
    .withColumn("days_since_prev", F.datediff("order_date", "prev_order_date"))
    .orderBy("customer_id", "order_date").show())

**Q33.** ⭐⭐  Split employees into **4 salary quartiles** with `ntile` and also compute `percent_rank`.

In [0]:
w = Window.orderBy("salary")
(employees.withColumn("quartile", F.ntile(4).over(w))
    .withColumn("pct_rank", F.round(F.percent_rank().over(w), 2))
    .select("emp_name", "salary", "quartile", "pct_rank").show())

**Q34.** ⭐⭐  Rewrite the **top-2 per department** problem in **Spark SQL** using a CTE (temp views already exist).

In [0]:
spark.sql('''
WITH ranked AS (
  SELECT e.*, DENSE_RANK() OVER (PARTITION BY dept_id ORDER BY salary DESC) AS drnk
  FROM employees e
  WHERE dept_id IS NOT NULL
)
SELECT * FROM ranked WHERE drnk <= 2 ORDER BY dept_id, drnk
''').show()

## 5 · Null handling & data cleaning (`dirty_customers`)

**Q35.** ⭐⭐  Count **NULLs in every column** dynamically (no hard-coded column names). Bonus: also count empty strings.

In [0]:
dirty_customers.select(
    [F.sum(F.col(c).isNull().cast("int")).alias(c) for c in dirty_customers.columns]
).show()

# nulls + empty strings
dirty_customers.select(
    [F.sum((F.col(c).isNull() | (F.col(c).cast("string") == "")).cast("int")).alias(c)
     for c in dirty_customers.columns]
).show()

**Q36.** ⭐  Fill NULL `name` with 'Unknown' and NULL `phone` with 'NA' using `fillna`; then do the same with `coalesce`. Convert empty-string emails to NULL.

In [0]:
dirty_customers.fillna({"name": "Unknown", "phone": "NA"}).show()

(dirty_customers
    .withColumn("name", F.coalesce("name", F.lit("Unknown")))
    .withColumn("email", F.when(F.trim("email") == "", None).otherwise(F.col("email")))
    .show())

**Q37.** ⭐⭐  Clean the data: **trim + collapse spaces + initcap** on `name`, **lowercase** `email`, keep only the **last 10 digits** of `phone`.

In [0]:
cleaned = (dirty_customers
    .withColumn("name", F.initcap(F.regexp_replace(F.trim("name"), r"\s+", " ")))
    .withColumn("email", F.lower(F.trim("email")))
    .withColumn("phone", F.substring(F.regexp_replace("phone", r"\D", ""), -10, 10)))
cleaned.show(truncate=False)

**Q38.** ⭐⭐⭐  `dob` has **mixed formats** (`dd/MM/yyyy`, `yyyy-MM-dd`, `dd-MMM-yyyy`, `yyyy/MM/dd`). Parse them into a single DATE column without failing on bad values.

In [0]:
fmts = ["dd/MM/yyyy", "yyyy-MM-dd", "dd-MMM-yyyy", "yyyy/MM/dd"]
# try_to_timestamp returns NULL instead of raising (important when ANSI mode is on, the default on newer Databricks runtimes)
dob_parsed = F.coalesce(*[F.expr(f"try_to_timestamp(dob, '{f}')") for f in fmts]).cast("date")
dirty_customers.withColumn("dob_date", dob_parsed).select("id", "dob", "dob_date").show()

**Q39.** ⭐⭐  Flag **valid emails** with a regex and extract the **domain**.

In [0]:
pattern = r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9-]+(\.[A-Za-z0-9-]+)*\.[A-Za-z]{2,}$"
(dirty_customers
    .withColumn("is_valid_email", F.col("email").rlike(pattern))
    .withColumn("domain", F.regexp_extract("email", r"@(.+)$", 1))
    .select("id", "email", "is_valid_email", "domain").show(truncate=False))

**Q40.** ⭐⭐  Remove **exact duplicate rows**, then keep only the **latest record per `id`** (by `updated_at`) using a window (not `dropDuplicates`).

In [0]:
print("exact dup removal:", dirty_customers.dropDuplicates().count())

w = Window.partitionBy("id").orderBy(F.col("updated_at").desc())
(dirty_customers.withColumn("rn", F.row_number().over(w))
    .filter("rn = 1").drop("rn").orderBy("id").show())
# dropDuplicates(["id"]) would keep an ARBITRARY row – that's why the window approach is preferred.

## 6 · Dates & strings

**Q41.** ⭐  From `orders` derive: day name, quarter, week-of-year and an `is_weekend` flag.

In [0]:
(orders.select(
    "order_id", "order_date",
    F.date_format("order_date", "EEEE").alias("day_name"),
    F.quarter("order_date").alias("quarter"),
    F.weekofyear("order_date").alias("week"),
    F.dayofweek("order_date").isin(1, 7).alias("is_weekend"),    # 1 = Sunday, 7 = Saturday
).show())

**Q42.** ⭐⭐  Taking **2024-03-01** as 'today': customers who signed up in the **last 12 months**, plus `months_since_signup`.

In [0]:
ref = F.lit("2024-03-01").cast("date")
(customers.filter(F.col("signup_date") >= F.add_months(ref, -12))
    .withColumn("months_since_signup", F.floor(F.months_between(ref, "signup_date")))
    .show())

**Q43.** ⭐⭐⭐  Build a **date calendar** for January 2024 (`sequence` + `explode`) and left-join daily revenue so days with no orders show **0**.

In [0]:
cal = spark.sql(
    "SELECT explode(sequence(to_date('2024-01-01'), to_date('2024-01-31'), interval 1 day)) AS dt"
)
daily = orders.groupBy("order_date").agg(F.sum("amount").alias("revenue"))
(
    cal.join(daily, cal.dt == daily.order_date, "left")
    .select("dt", F.coalesce("revenue", F.lit(0.0)).alias("revenue"))
    .orderBy("dt")
    .show(31)
)

**Q44.** ⭐⭐  String functions: extract **username** from email, build an order code like **ORD00000101** (`lpad`), and get the first letter + length of `customer_name`.

In [0]:
(customers.select(
    "customer_name",
    F.split("email", "@")[0].alias("username"),
    F.regexp_extract("email", r"@([^.]+)\.", 1).alias("provider"),
    F.upper(F.substring("customer_name", 1, 1)).alias("initial"),
    F.length("customer_name").alias("name_len"),
).show())

orders.select(F.concat(F.lit("ORD"), F.lpad(F.col("order_id").cast("string"), 8, "0")).alias("order_code")).show(3)

## 7 · Complex types: JSON, arrays, structs, maps, pivot

**Q45.** ⭐⭐⭐  Parse the JSON `payload` with an explicit schema (`from_json`), then **explode the items** so there is one row per item with `order_id`, `customer_name`, `sku`, `qty`.

In [0]:
schema = (
    "order_id INT, customer STRUCT<id: INT, name: STRING>, "
    "items ARRAY<STRUCT<sku: STRING, qty: INT>>, tags ARRAY<STRING>"
)
parsed = json_events.withColumn("j", F.from_json("payload", schema))
(
    parsed.select(
        "j.order_id",
        F.col("j.customer.name").alias("customer_name"),
        F.explode("j.items").alias("item"),
    )
    .select("order_id", "customer_name", "item.sku", "item.qty")
    .show()
)

**Q46.** ⭐⭐  Without a schema: extract `customer.name` and the first item's `sku` with `get_json_object`, and top-level fields with `json_tuple`.

In [0]:
json_events.select(
    "id",
    F.get_json_object("payload", "$.customer.name").alias("customer_name"),
    F.get_json_object("payload", "$.items[0].sku").alias("first_sku"),
).show()

json_events.select("id", F.json_tuple("payload", "order_id", "tags").alias("order_id", "tags_json")).show()

**Q47.** ⭐⭐⭐  Array functions: `size`, `array_contains` ('vip'), **total qty** via the higher-order `aggregate`, qty list via `transform`, and `posexplode` on tags.

In [0]:
schema = ("order_id INT, customer STRUCT<id: INT, name: STRING>, "
          "items ARRAY<STRUCT<sku: STRING, qty: INT>>, tags ARRAY<STRING>")
p = json_events.withColumn("j", F.from_json("payload", schema))
p.select(
    "id",
    F.size("j.items").alias("n_items"),
    F.array_contains("j.tags", "vip").alias("is_vip"),
    F.aggregate("j.items", F.lit(0), lambda acc, x: acc + x["qty"]).alias("total_qty"),
    F.transform("j.items", lambda x: x["qty"]).alias("qty_list"),
).show()

p.select("id", F.posexplode("j.tags").alias("pos", "tag")).show()

**Q48.** ⭐⭐  Build a **map column** (`city`, `dept`) per employee; read a key, list the keys, and explode the map.

In [0]:
m = employees.select(
    "emp_name",
    F.create_map(
        F.lit("city"), F.col("city"), F.lit("dept"), F.col("dept_id").cast("string")
    ).alias("attrs"),
)
m.select(
    "emp_name", F.col("attrs")["city"].alias("city"), F.map_keys("attrs").alias("keys")
).show(3)
m.select("emp_name", F.explode("attrs").alias("k", "v")).show(6)

**Q49.** ⭐  Convert every employee row into a **JSON string** (`to_json(struct(...))`).

In [0]:
employees.select(F.to_json(F.struct("*")).alias("json")).show(3, truncate=False)

**Q50.** ⭐⭐⭐  **Pivot** DELIVERED revenue so each month is a column per customer, then **unpivot** it back to long form.

In [0]:
months = ["2024-01", "2024-02", "2024-03", "2024-04", "2024-05", "2024-06"]
pivoted = (
    orders.filter(F.col("status") == "DELIVERED")
    .withColumn("month", F.date_format("order_date", "yyyy-MM"))
    .groupBy("customer_id")
    .pivot("month", months)
    .agg(F.sum("amount"))
    .orderBy("customer_id")
)
pivoted.show()

(
    pivoted.unpivot("customer_id", months, "month", "revenue")
    .filter(F.col("revenue").isNotNull())
    .orderBy("customer_id", "month")
    .show()
)